# Weekly Feature Engineering Dataset

Build one weekly dataset joining resident, vitals, and claims signals.

Leakage control:
- targets are shifted to the **next week** per resident
- current-week incident/claim aggregates are dropped from features
- features remain from the **current week**

Output:
- `data/weekly_model_dataset.parquet`


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path('..').resolve() / 'data'
OUT_DIR = BASE_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_START = pd.Timestamp('2023-10-01')
DATE_END = pd.Timestamp('2024-12-30')


In [2]:
def pick_col(df, candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f'Missing required column among: {candidates}')
    return None


def safe_join_unique(values, unknown='Unknown'):
    vals = [str(x).strip() for x in values.dropna().astype(str) if str(x).strip()]
    vals = [v for v in vals if v.lower() != 'nan']
    uniq = sorted(set(vals))
    return '|'.join(uniq) if uniq else unknown


## Resident Weekly Base (Includes Age)


In [3]:
residents = pd.read_parquet(BASE_DIR / 'residents.parquet').copy()
for c in ['admission_date', 'discharge_date', 'deceased_date', 'date_of_birth']:
    if c in residents.columns:
        residents[c] = pd.to_datetime(residents[c], errors='coerce')

if 'date_of_birth' not in residents.columns:
    residents['date_of_birth'] = pd.NaT

base = residents.dropna(subset=['admission_date']).copy()

cutoff = pd.concat([
    base['admission_date'],
    base['discharge_date'] if 'discharge_date' in base.columns else pd.Series(dtype='datetime64[ns]'),
    base['deceased_date'] if 'deceased_date' in base.columns else pd.Series(dtype='datetime64[ns]'),
], ignore_index=True).max()

end_candidates = [c for c in ['discharge_date', 'deceased_date'] if c in base.columns]
base['end_date'] = base[end_candidates].min(axis=1) if end_candidates else pd.NaT
base['end_date'] = base['end_date'].fillna(cutoff)
base = base[base['end_date'] >= base['admission_date']].copy()

rows = []
for r in base[['resident_id', 'facility_id', 'admission_date', 'end_date', 'date_of_birth']].itertuples(index=False):
    for p in pd.period_range(r.admission_date, r.end_date, freq='W-SUN'):
        week_start = p.start_time
        week_end = p.end_time.floor('D')
        age_years = np.nan
        if pd.notna(r.date_of_birth):
            age_years = (week_start - r.date_of_birth).days / 365.25
        rows.append((r.resident_id, r.facility_id, week_start, week_end, age_years))

resident_weekly = pd.DataFrame(rows, columns=['resident_id', 'facility_id', 'week_start', 'week_end', 'resident_age'])
resident_weekly = resident_weekly.sort_values(['resident_id', 'week_start']).reset_index(drop=True)
resident_weekly.head()


,resident_id,facility_id,week_start,week_end,resident_age
0,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-03-18,2024-03-24,62.847365
1,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-03-25,2024-03-31,62.866530
2,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-04-01,2024-04-07,62.885695
3,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-04-08,2024-04-14,62.904860
4,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-04-15,2024-04-21,62.924025


## Vitals Weekly Features


In [4]:
vitals = pd.read_parquet(BASE_DIR / 'vitals.parquet').copy()
time_col = pick_col(vitals, ['measured_at', 'created_at'], required=True)

vitals[time_col] = pd.to_datetime(vitals[time_col], errors='coerce')
vitals = vitals.dropna(subset=['resident_id', time_col, 'vital_type'])
vitals['value'] = pd.to_numeric(vitals['value'], errors='coerce')
if 'dystolic_value' not in vitals.columns:
    vitals['dystolic_value'] = np.nan
vitals['dystolic_value'] = pd.to_numeric(vitals['dystolic_value'], errors='coerce')

vitals_pivot = (
    vitals
    .pivot_table(
        index=['resident_id', time_col],
        columns='vital_type',
        values='value',
        aggfunc='mean'
    )
    .reset_index()
)

dystolic_at_time = (
    vitals
    .groupby(['resident_id', time_col], as_index=False)['dystolic_value']
    .mean()
)

vitals_pivot = vitals_pivot.merge(dystolic_at_time, on=['resident_id', time_col], how='left')
vitals_pivot = vitals_pivot.rename(columns={time_col: 'event_at'})
vitals_pivot['week_start'] = vitals_pivot['event_at'].dt.to_period('W').dt.start_time

metric_cols = [c for c in vitals_pivot.columns if c not in ['resident_id', 'event_at', 'week_start']]
agg_map = {c: ['mean', 'std'] for c in metric_cols if c != 'dystolic_value'}
agg_map['dystolic_value'] = ['mean']

vitals_weekly = vitals_pivot.groupby(['resident_id', 'week_start']).agg(agg_map)
vitals_weekly.columns = [f'{c}_{s}' for c, s in vitals_weekly.columns]
vitals_weekly = vitals_weekly.reset_index().sort_values(['resident_id', 'week_start'])
vitals_weekly.head()


,resident_id,week_start,BP - Systolic_mean,BP - Systolic_std,Blood Sugar_mean,Blood Sugar_std,O2 sats_mean,O2 sats_std,Pain Level_mean,Pain Level_std,Pulse_mean,Pulse_std,Respiration_mean,Respiration_std,Temperature_mean,Temperature_std,Weight_mean,Weight_std,dystolic_value_mean
0,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-03-18,129.230769,9.337682,NaN,NaN,94.000000,10.255080,0.000000,0.000000,66.000000,7.141428,18.153846,0.688737,97.753846,0.352646,254.2,36.48671,75.384615
1,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-03-25,129.944444,7.132958,NaN,NaN,96.555556,1.149026,1.000000,1.818706,69.444444,6.500880,18.277778,0.669113,97.911111,0.443103,228.2,NaN,73.833333
2,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-04-01,123.850000,11.485804,NaN,NaN,96.050000,1.394538,0.380952,1.745743,63.300000,8.663900,17.950000,0.510418,97.760000,0.434560,NaN,NaN,75.450000
3,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-04-08,127.200000,9.361736,NaN,NaN,97.000000,0.794719,0.181818,0.852803,61.631579,4.716526,17.789474,0.418854,97.840000,0.395235,223.0,NaN,75.150000
4,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-04-15,124.190476,10.604806,NaN,NaN,96.523810,0.679636,0.291667,0.999094,66.714286,8.492013,17.952381,0.384212,98.047619,0.422633,NaN,NaN,73.857143


## Claims Weekly Features


In [5]:
incidents = pd.read_parquet(BASE_DIR / 'incidents.parquet').copy()
injuries = pd.read_parquet(BASE_DIR / 'injuries.parquet').copy()
adm = pd.read_parquet(BASE_DIR / 'hospital_admissions.parquet').copy()
trf = pd.read_parquet(BASE_DIR / 'hospital_transfers.parquet').copy()

inc_time_col = pick_col(incidents, ['occurred_at', 'reported_at', 'created_at'], required=True)
incidents['incident_at'] = pd.to_datetime(incidents[inc_time_col], errors='coerce')
incidents = incidents.dropna(subset=['incident_id', 'resident_id', 'incident_at']).copy()
if 'incident_type' not in incidents.columns:
    incidents['incident_type'] = 'Unknown'
incidents['incident_type'] = incidents['incident_type'].fillna('Unknown').astype(str).str.strip().replace('', 'Unknown')

inj_id_col = pick_col(injuries, ['injury_id'])
if inj_id_col is None:
    injuries = injuries.reset_index(drop=False).rename(columns={'index': 'injury_id'})
    inj_id_col = 'injury_id'

if 'incident_id' in injuries.columns:
    inj_agg = (
        injuries[['incident_id', inj_id_col]]
        .dropna(subset=['incident_id'])
        .groupby('incident_id', as_index=False)
        .agg(injury_count=(inj_id_col, 'nunique'))
    )
else:
    inj_agg = pd.DataFrame(columns=['incident_id', 'injury_count'])


def hospital_link_count(events_df, id_candidates, eff_candidates, inef_candidates):
    e = events_df.copy()
    id_col = pick_col(e, id_candidates)
    if id_col is None:
        e = e.reset_index(drop=False).rename(columns={'index': 'event_id'})
        id_col = 'event_id'
    if 'resident_id' not in e.columns:
        return pd.DataFrame(columns=['incident_id', 'count', 'first_event_at'])

    eff_col = pick_col(e, eff_candidates)
    if eff_col is None:
        return pd.DataFrame(columns=['incident_id', 'count', 'first_event_at'])
    inef_col = pick_col(e, inef_candidates)

    e['event_id'] = e[id_col]
    e['event_effective_at'] = pd.to_datetime(e[eff_col], errors='coerce')
    e['event_ineffective_at'] = pd.to_datetime(e[inef_col], errors='coerce') if inef_col else pd.NaT
    if 'facility_id' not in e.columns:
        e['facility_id'] = pd.NA

    e = e[['event_id', 'resident_id', 'facility_id', 'event_effective_at', 'event_ineffective_at']].dropna(subset=['resident_id', 'event_effective_at'])

    m = incidents[['incident_id', 'resident_id', 'facility_id', 'incident_at']].merge(e, on='resident_id', how='left', suffixes=('_inc', '_evt'))
    fac_ok = m['facility_id_inc'].isna() | m['facility_id_evt'].isna() | (m['facility_id_inc'] == m['facility_id_evt'])
    interval_contains = (m['event_effective_at'] <= m['incident_at']) & (m['event_ineffective_at'].isna() | (m['event_ineffective_at'] >= m['incident_at']))
    post_7d = (m['event_effective_at'] >= m['incident_at']) & (m['event_effective_at'] <= m['incident_at'] + pd.Timedelta(days=7))

    m = m[fac_ok & (interval_contains | post_7d)]
    if m.empty:
        return pd.DataFrame(columns=['incident_id', 'count', 'first_event_at'])
    return m.groupby('incident_id', as_index=False).agg(count=('event_id', 'nunique'), first_event_at=('event_effective_at', 'min'))

adm_agg = hospital_link_count(
    adm,
    ['admission_id', 'hospital_admission_id'],
    ['effective_date', 'admitted_at', 'start_at', 'created_at'],
    ['ineffective_date', 'discharged_at', 'end_at']
).rename(columns={'count': 'admission_count', 'first_event_at': 'first_admission_at'})

trf_agg = hospital_link_count(
    trf,
    ['transfer_id', 'hospital_transfer_id'],
    ['effective_date', 'transferred_at', 'start_at', 'created_at'],
    ['ineffective_date', 'end_at']
).rename(columns={'count': 'transfer_count', 'first_event_at': 'first_transfer_at'})

claims_incident = (
    incidents[['incident_id', 'resident_id', 'facility_id', 'incident_at', 'incident_type']]
    .merge(inj_agg, on='incident_id', how='left')
    .merge(adm_agg, on='incident_id', how='left')
    .merge(trf_agg, on='incident_id', how='left')
)

for c in ['injury_count', 'admission_count', 'transfer_count']:
    claims_incident[c] = claims_incident[c].fillna(0).astype('int64')

claims_incident['had_injury'] = (claims_incident['injury_count'] > 0).astype('int8')
claims_incident['had_admission'] = (claims_incident['admission_count'] > 0).astype('int8')
claims_incident['had_transfer'] = (claims_incident['transfer_count'] > 0).astype('int8')
claims_incident['is_claim'] = 1

claims_incident['week_start'] = claims_incident['incident_at'].dt.to_period('W').dt.start_time
claims_incident['week_end'] = claims_incident['incident_at'].dt.to_period('W').dt.end_time.dt.floor('D')

claims_weekly = (
    claims_incident
    .groupby(['resident_id', 'week_start', 'week_end'], as_index=False)
    .agg(
        claim_count=('incident_id', 'nunique'),
        incident_type_list=('incident_type', safe_join_unique),
        incident_type_primary=('incident_type', lambda s: s.value_counts().index[0] if len(s) else 'Unknown'),
        injury_count_sum=('injury_count', 'sum'),
        admission_count_sum=('admission_count', 'sum'),
        transfer_count_sum=('transfer_count', 'sum'),
        had_injury_week=('had_injury', 'max'),
        had_admission_week=('had_admission', 'max'),
        had_transfer_week=('had_transfer', 'max'),
        is_claim_week=('is_claim', 'max'),
    )
    .sort_values(['resident_id', 'week_start'])
    .reset_index(drop=True)
)
claims_weekly.head()


,resident_id,week_start,week_end,claim_count,incident_type_list,incident_type_primary,injury_count_sum,admission_count_sum,transfer_count_sum,had_injury_week,had_admission_week,had_transfer_week,is_claim_week
0,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-05-27,2024-06-02,1,Wound,Wound,0,0,0,0,0,0,1
1,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-09-23,2024-09-29,1,Wound,Wound,1,0,0,1,0,0,1
2,00a15cf8-e353-541d-8896-e18ef59e746d,2024-05-20,2024-05-26,1,Altercation,Altercation,0,0,0,0,0,0,1
3,00a15cf8-e353-541d-8896-e18ef59e746d,2024-11-25,2024-12-01,1,Fall,Fall,0,0,0,0,0,0,1
4,00c0fe34-80a2-518f-9d69-dc34e040911b,2023-08-07,2023-08-13,1,Wound,Wound,1,0,0,1,0,0,1


## Join, Create Next-Week Targets, Save Final Dataset


In [6]:
weekly_dataset = (
    resident_weekly
    .merge(vitals_weekly, on=['resident_id', 'week_start'], how='left')
    .merge(claims_weekly, on=['resident_id', 'week_start', 'week_end'], how='left')
    .sort_values(['resident_id', 'week_start'])
    .reset_index(drop=True)
)

# Fill no-claim weeks for current-week claim features (used only to build NEXT-WEEK targets)
weekly_dataset['claim_count'] = weekly_dataset['claim_count'].fillna(0).astype('int64')
weekly_dataset['is_claim_week'] = weekly_dataset['is_claim_week'].fillna(0).astype('int8')
weekly_dataset['incident_type_primary'] = weekly_dataset['incident_type_primary'].fillna('NoClaim')
weekly_dataset['incident_type_list'] = weekly_dataset['incident_type_list'].fillna('NoClaim')

# Restrict time range
weekly_dataset = weekly_dataset[(weekly_dataset['week_start'] >= DATE_START) & (weekly_dataset['week_start'] <= DATE_END)].copy()

# Leakage-safe targets: NEXT WEEK labels
weekly_dataset = weekly_dataset.sort_values(['resident_id', 'week_start']).reset_index(drop=True)
weekly_dataset['target_claim_next_week'] = (
    weekly_dataset.groupby('resident_id')['claim_count'].shift(-1).fillna(0) > 0
).astype('int8')
weekly_dataset['target_claim_type_next_week'] = (
    weekly_dataset.groupby('resident_id')['incident_type_primary'].shift(-1).fillna('NoClaim')
)

# Remove rows without a true future week in-range for same resident
has_future = weekly_dataset.groupby('resident_id')['week_start'].transform('max') > weekly_dataset['week_start']
weekly_dataset_final = weekly_dataset[has_future].copy()

# Strict anti-leakage: drop current-week columns derived from incidents/claims
incident_current_week_cols = [
    'claim_count',
    'incident_type_list',
    'incident_type_primary',
    'injury_count_sum',
    'admission_count_sum',
    'transfer_count_sum',
    'had_injury_week',
    'had_admission_week',
    'had_transfer_week',
    'is_claim_week',
]
drop_cols = [c for c in incident_current_week_cols if c in weekly_dataset_final.columns]
weekly_dataset_final = weekly_dataset_final.drop(columns=drop_cols)

out_path = OUT_DIR / 'weekly_model_dataset.parquet'
weekly_dataset_final.to_parquet(out_path, index=False)

print('Saved:', out_path)
print('Rows:', len(weekly_dataset_final))
print('Residents:', weekly_dataset_final['resident_id'].nunique())
print('Date range:', weekly_dataset_final['week_start'].min(), '->', weekly_dataset_final['week_start'].max())
print('Target prevalence:', weekly_dataset_final['target_claim_next_week'].mean())
print('Dropped leakage columns:', drop_cols)
weekly_dataset_final.head()



Saved: /home/rubim/Documents/Projects/repos/tricura_case/data/weekly_model_dataset.parquet
Rows: 53985
Residents: 2559
Date range: 2023-10-02 00:00:00 -> 2024-12-23 00:00:00
Target prevalence: 0.035343150875243125
Dropped leakage columns: ['claim_count', 'incident_type_list', 'incident_type_primary', 'injury_count_sum', 'admission_count_sum', 'transfer_count_sum', 'had_injury_week', 'had_admission_week', 'had_transfer_week', 'is_claim_week']


,resident_id,facility_id,week_start,week_end,resident_age,BP - Systolic_mean,BP - Systolic_std,Blood Sugar_mean,Blood Sugar_std,O2 sats_mean,...,Pulse_std,Respiration_mean,Respiration_std,Temperature_mean,Temperature_std,Weight_mean,Weight_std,dystolic_value_mean,target_claim_next_week,target_claim_type_next_week
0,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-03-18,2024-03-24,62.847365,129.230769,9.337682,NaN,NaN,94.000000,...,7.141428,18.153846,0.688737,97.753846,0.352646,254.2,36.48671,75.384615,0,NoClaim
1,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-03-25,2024-03-31,62.866530,129.944444,7.132958,NaN,NaN,96.555556,...,6.500880,18.277778,0.669113,97.911111,0.443103,228.2,NaN,73.833333,0,NoClaim
2,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-04-01,2024-04-07,62.885695,123.850000,11.485804,NaN,NaN,96.050000,...,8.663900,17.950000,0.510418,97.760000,0.434560,NaN,NaN,75.450000,0,NoClaim
3,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-04-08,2024-04-14,62.904860,127.200000,9.361736,NaN,NaN,97.000000,...,4.716526,17.789474,0.418854,97.840000,0.395235,223.0,NaN,75.150000,0,NoClaim
4,000d4288-4983-5512-b4e8-bf5efa8e20ef,75713da7-fe55-5b40-b6db-01ad1c35a615,2024-04-15,2024-04-21,62.924025,124.190476,10.604806,NaN,NaN,96.523810,...,8.492013,17.952381,0.384212,98.047619,0.422633,NaN,NaN,73.857143,0,NoClaim
